## Setup

In [1]:
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO_ROOT = Path.cwd().resolve()
print(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.append("..")

/Users/tzhang04/Desktop/active-passive-alternations/notebooks


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import normaltest, ttest_rel, wilcoxon
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display
from src.uid import *

/Users/tzhang04/Desktop/active-passive-alternations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-03 20:23:35 INFO: Downloaded file to /Users/tzhang04/stanza_resources/resources.json
2026-04-03 20:23:35 INFO: Downloading default packages for language: en (English) ...
2026-04-03 20:23:36 INFO: File exists: /Users/tzhang04/stanza_resources/en/default.zip
2026-04-03 20:23:38 INFO: Finished downloading models and saved to /Users/tzhang04/stanza_resources
2026-04-03 20:23:38 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-03 20:23:38 INFO: Downloaded file to /Users/tzhang04/stanza_resources/resources.json
2026-04-03 20:23:39 INFO: Loading t

In [3]:
import conllu
from conllu import Token, parse, parse_tree
from pyinflect import getAllInflections, getInflection
from src.utils import *
from src.units.word import *
from src.units.sentence import *

## Test Feature Extraction

In [4]:
ud_path = "../data/en_gum-ud-dev.conllu"
docs = iter_counterfactual_docs(ud_path)

In [107]:
doc = [doc for doc in docs if doc[0] == 'cf::GUM_conversation_grounded::22::a>p'][0]
key_sents = []
for doc in docs:
    for sent in doc[2]:
        if " her " in sent.text:
            print("Found it")
            key_sents.append(sent)

Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
Found it
F

In [108]:
[sent.text for sent in key_sents]

["And her mom wanted to take her home early, and I'm like, no let's stay longer.",
 "But her mom wouldn't let her.",
 "You'll get a hold of her first.",
 "I didn't want you to have her phone number.",
 "See now she's trying to think of ways to cover her tracks.",
 "And her mom wanted to take her home early, and I'm like, no let's stay longer.",
 "But her mom wouldn't let her.",
 "You'll get a hold of her first.",
 "I didn't want you to have her phone number.",
 "See now she's trying to think of ways to cover her tracks.",
 "And her mom wanted to take her home early, and I'm like, no let's stay longer.",
 "But her mom wouldn't let her.",
 "You'll get a hold of her first.",
 "I didn't want you to have her phone number.",
 "See now she's trying to think of ways to cover her tracks.",
 "And her mom wanted to take her home early, and I'm like, no let's stay longer.",
 "But her mom wouldn't let her.",
 "You'll get a hold of her first.",
 "I didn't want you to have her phone number.",
 "See n

In [109]:
key_sents[0]

[{'id': 1,
  'form': 'And',
  'lemma': 'and',
  'upos': 'CCONJ',
  'xpos': 'CC',
  'feats': None,
  'head': 4,
  'deprel': 'cc',
  'deps': [('cc', 4)],
  'misc': {'Discourse': 'joint-list_m:39->38:1:sem-lxchn-193,206-_+dm-and-199-_',
   'PDTB': 'Explicit:Expansion.Conjunction:and:199:183-198:200-219'},
  'inflection': 'CC',
  'children': []},
 {'id': 2,
  'form': 'her',
  'lemma': 'her',
  'upos': 'PRON',
  'xpos': 'PRP$',
  'feats': {'Case': 'Gen',
   'Gender': 'Fem',
   'Number': 'Sing',
   'Person': '3',
   'Poss': 'Yes',
   'PronType': 'Prs'},
  'head': 3,
  'deprel': 'nmod:poss',
  'deps': [('nmod:poss', 3)],
  'misc': {'Entity': '(22-person-new-nnnnn-cf4-2-coref(17-person-giv:act-nnnnn-cf2-1-ana)'},
  'inflection': 'PRP$',
  'children': []},
 {'id': 3,
  'form': 'mom',
  'lemma': 'mom',
  'upos': 'NOUN',
  'xpos': 'NN',
  'feats': {'Number': 'Sing'},
  'head': 4,
  'deprel': 'nsubj',
  'deps': [('nsubj', 4), ('nsubj:xsubj', 6)],
  'misc': {'Entity': '22)'},
  'inflection': 'NN',


In [80]:
nlp = stanza.MultilingualPipeline()

2026-04-03 19:52:26 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-03 19:52:26 INFO: Downloaded file to /Users/tzhang04/stanza_resources/resources.json
2026-04-03 19:52:26 INFO: Loading these models for language: multilingual ():
| Processor | Package |
-----------------------
| langid    | ud      |

2026-04-03 19:52:26 INFO: Using device: cpu
2026-04-03 19:52:26 INFO: Loading: langid
2026-04-03 19:52:26 INFO: Done loading processors!


In [94]:
nlp("My mom picked up a dog in the park.")

[
  [
    {
      "id": 1,
      "text": "My",
      "lemma": "my",
      "upos": "PRON",
      "xpos": "PRP$",
      "feats": "Case=Gen|Number=Sing|Person=1|Poss=Yes|PronType=Prs",
      "head": 2,
      "deprel": "nmod:poss",
      "start_char": 0,
      "end_char": 2,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 2,
      "text": "mom",
      "lemma": "mom",
      "upos": "NOUN",
      "xpos": "NN",
      "feats": "Number=Sing",
      "head": 3,
      "deprel": "nsubj",
      "start_char": 3,
      "end_char": 6,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 3,
      "text": "picked",
      "lemma": "pick",
      "upos": "VERB",
      "xpos": "VBD",
      "feats": "Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin",
      "head": 0,
      "deprel": "root",
      "start_char": 7,
      "end_char": 13,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 4,
      "text": "up"

In [6]:
doc_id, sents, doc = docs[0]

In [7]:
for doc_id, sents, doc in docs:
    for sent in doc:
        try:
            active_sent = ActiveSentence(sent)
        except:
            continue

In [8]:
active_sent.text

'If you wash your overalls alone or in a light load, use about half the detergent called for and less water.'

In [9]:
active_sent.active_subject_word

{'id': 2,
 'form': 'you',
 'lemma': 'you',
 'upos': 'PRON',
 'xpos': 'PRP',
 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '2', 'PronType': 'Prs'},
 'head': 3,
 'deprel': 'nsubj',
 'deps': [('nsubj', 3)],
 'misc': {'Entity': '(3-person-giv:inact-nnsnn-cf1-1-ana)'},
 'inflection': 'PRP',
 'children': []}

In [10]:
tok, model, device = load_lm('distilgpt2', device='mps')
unigram = UnigramLM(tok)
unigram.fit("\n".join(["\n".join(doc[1]) for doc in docs]), uid_unit='word')

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2572.36it/s, Materializing param=transformer.wte.weight]            
GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (487324 > 1024). Running this sequence through the model will result in indexing errors


In [11]:
result = ling_features(active_sent, tok, unigram, uid_unit='word')

In [12]:
unigram.model.get('you', 0)

0.01065558639014376

In [13]:
unigram('you')

(array([0.01065559]), np.float64(0.01065558639014376))

In [15]:
result

{'agent': [{'id': 2,
   'form': 'you',
   'lemma': 'you',
   'upos': 'PRON',
   'xpos': 'PRP',
   'feats': {'Case': 'Nom',
    'Number': 'Sing',
    'Person': '2',
    'PronType': 'Prs'},
   'head': 3,
   'deprel': 'nsubj',
   'deps': [('nsubj', 3)],
   'misc': {'Entity': '(3-person-giv:inact-nnsnn-cf1-1-ana)'},
   'inflection': 'PRP',
   'children': []}],
 'agent_len': 1,
 'agent_unigram_prob': np.float64(0.01065558639014376),
 'agent_is_pronoun': True,
 'agent_is_plural': False,
 'patient': [{'id': 4,
   'form': 'your',
   'lemma': 'your',
   'upos': 'PRON',
   'xpos': 'PRP$',
   'feats': {'Case': 'Gen',
    'Number': 'Sing',
    'Person': '2',
    'Poss': 'Yes',
    'PronType': 'Prs'},
   'head': 5,
   'deprel': 'nmod:poss',
   'deps': [('nmod:poss', 5)],
   'misc': {'Entity': '(1-object-giv:inact-sssss-cf2-2-coref(3-person-giv:act-nnsnn-cf1-1-ana)'},
   'inflection': 'PRP$',
   'children': []},
  {'id': 5,
   'form': 'overalls',
   'lemma': 'overall',
   'upos': 'NOUN',
   'xpos': 

In [5]:
# Animacy from WordNet
import nltk
nltk.download("wordnet")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/tzhang04/nltk_data...


In [91]:
syn = wn.synsets("we", pos=wn.NOUN)
i = 0
syn[i], syn[i].lexname(), syn[i].definition()

IndexError: list index out of range

In [92]:
syn

[]

In [90]:
is_animate("we")

False

#### Check outputs

In [110]:
out = pd.read_csv("../temp/cf_word_sentence_uid_chkpt.csv")

In [125]:
ling_features = ["doc_id", "sent_idx"]
for name in ['agent', 'patient']:
    ling_features.extend([
        f"{name}",
        f"{name}_len", 
        f"{name}_unigram_logprob",
        f"{name}_is_pronoun",
        f"{name}_is_plural",
        f"{name}_is_animate",
        f"{name}_is_definite",])

In [126]:
ling_feat_out = out[['sentence'] + ling_features]

In [127]:
ling_feat_out[ling_feat_out['agent_is_definite']]

,sentence,doc_id,sent_idx,agent,agent_len,agent_unigram_logprob,agent_is_pronoun,agent_is_plural,agent_is_animate,agent_is_definite,patient,patient_len,patient_unigram_logprob,patient_is_pronoun,patient_is_plural,patient_is_animate,patient_is_definite
8,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
9,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
10,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
11,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
12,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
13,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
14,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
15,Such a scenario may be found in different situ...,f::GUM_academic_exposure::-1::og,6,one,1,-4.739847,True,False,True,True,a language,2,-8.681186,False,False,False,False
16,"In the present study, we examine the outcomes ...",f::GUM_academic_exposure::-1::og,7,we,1,-5.501987,True,True,True,True,the outcomes of such a period of no exposure o...,15,-65.177692,False,True,False,True
17,"In the present study, we examine the outcomes ...",f::GUM_academic_exposure::-1::og,7,we,1,-5.501987,True,True,True,True,the outcomes of such a period of no exposure o...,15,-65.177692,False,True,False,True


In [5]:
doc = [doc for doc in docs if doc[0] == 'f::GUM_academic_exposure::-1::og'][0]
is_definite(doc[2][13][18])

False

In [11]:
is_definite(doc[2][13].active_subject_word)

False